In [ ]:
# Filename: process_legal.ipynb
#
# ---------------------------------------------------------
# 1. CONFIGURATION
# 2. HELPERS
#   a. loading stopwords 
#   b. loading Thesaurus
#   c. verify/validate quantization before model load
#   d. model load from previous model save, if fails continue from scratch
# 3. RAG MANAGER
# 4. PIPELINE
#   a. Load and process text files in the directory
#   b. Summarization model on (CPU)
#   c. RAG + generate output based on prompt
# 5. MAIN FUNCTION
# ---------------------------------------------------------

import os, re, torch
torch.set_num_threads(12)

USE_CUDA = torch.cuda.is_available()

if USE_CUDA:
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

import pandas as pd
from datetime import datetime
from typing import List, Set

from langchain_community.document_loaders import WebBaseLoader, Docx2txtLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from striprtf.striprtf import rtf_to_text
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# 1. CONFIGURATION
CONFIG = {
            "mode": "title", # keyword or title
            "main_model_id": "Qwen/Qwen3-4B-Thinking-2507",
            #"main_model_id": "Qwen/Qwen3.5-9B",
            "summ_model_id": "slovak-nlp/mistral-sk-7b",
            "emb_model_id": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            "input_dir": "./Input",
            "output_dir": "./Output",
            "models_dir": "./Models"
            "rag_dir": "./RAG_store",
            "thesaurus_path": "./Thesaurus/SK_Local_Thesaurus.xlsx",
            "thesaurus_col": "slovak_terms",
            "stopwords_path": "./Input/stopword_dictionary.txt",
            "allowed_domains": ["slov-lex.sk", "justice.gov.sk", "najpravo.sk", "zakonypreludi.sk"],
            "chunk_size": 1200,
            "chunk_overlap": 200,
            "top_k": 4,
            }

for d in (CONFIG["output_dir"], CONFIG["rag_dir"], CONFIG["models_dir"]):
    os.makedirs(d, exist_ok=True)


# 2. HELPERS
def load_rtf(path: str) -> List[Document]:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return [Document(page_content=rtf_to_text(f.read()), metadata={"source": path})]
    except Exception as e:
        print(f"[WARN] RTF parsing failed for {path}: {e}")
        return []

# a. loading stopwords 
def load_stopwords(path: str) -> Set[str]:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip().lower() for line in f if line.strip()}

# b. loading Thesaurus
def load_thesaurus(path: str, col: str, stopwords: Set[str]) -> List[str]:
    if not os.path.exists(path):
        return []
    df = pd.read_excel(path, engine="openpyxl")
    if col not in df.columns:
        return []
    seen, terms = set(), []
    for cell in df[col].dropna().astype(str):
        for t in re.split(r"[,\n;]+", cell):
            t = t.strip()
            if len(t) > 1 and t.lower() not in stopwords and t not in seen:
                seen.add(t)
                terms.append(t)
    print(f"[THESAURUS] Loaded {len(terms)} terms.")
    return terms

def match_terms(text: str, terms: List[str], limit: int = 10) -> str:
    if not text or not terms:
        return ""
    hits = [t for t in terms if re.search(rf"\b{re.escape(t)}\b", text, re.IGNORECASE)]
    hits.sort(key=len, reverse=True)
    return ", ".join(hits[:limit])

# c. verify/validate quantization before model load
def _verify_quantization(quantize: str | None, device: str) -> str | None:
    if quantize is None:
        return None

    if device == "cpu":
        print(f"[QUANT] Quantization '{quantize}' not supported on CPU -> fallback to fp32.")
        return None

    if not torch.cuda.is_available():
        print(f"[QUANT] CUDA not available -> fallback to fp32.")
        return None

    try:
        import bitsandbytes as bnb
        print(f"[QUANT] bitsandbytes {bnb.__version__} detected.")
    except ImportError:
        print("[QUANT] bitsandbytes not installed -> fallback to fp16.")
        return None
    except Exception as e:
        print(f"[QUANT] bitsandbytes import failed: {e} -> fallback to fp16.")
        return None

    try:
        if quantize == "4bit":
            _ = BitsAndBytesConfig(
                                    load_in_4bit=True,
                                    bnb_4bit_quant_type="nf4",
                                    bnb_4bit_compute_dtype=torch.float16,
                                    bnb_4bit_use_double_quant=True,
                                    )
        elif quantize == "8bit":
            _ = BitsAndBytesConfig(load_in_8bit=True)
        else:
            print(f"[QUANT] Unknown quantization mode '{quantize}' -> fallback to fp16.")
            return None
    except Exception as e:
        print(f"[QUANT] BitsAndBytesConfig init failed: {e} -> fallback to fp16.")
        return None

    try:
        free_mem, total_mem = torch.cuda.mem_get_info()
        free_gb = free_mem / (1024 ** 3)
        print(f"[QUANT] CUDA free VRAM: {free_gb:.2f} GB")
        if free_gb < 2.0:
            print(f"[QUANT] Very low VRAM ({free_gb:.2f} GB) — quantization may fail at load.")
    except Exception:
        pass

    print(f"[QUANT] Verification OK for '{quantize}'.")
    return quantize

# d. model load from previous model save, if fails continue from scratch
def load_model(model_id: str, device: str, quantize: str | None = None):
    safe_name = model_id.replace("/", "_")
    local_path = os.path.join(CONFIG["models_dir"], safe_name)

    quantize = _verify_quantization(quantize, device)

    if os.path.isdir(local_path) and os.path.exists(os.path.join(local_path, "config.json")):
        source = local_path
        from_local = True
        print(f"[{device.upper()}] Loading {model_id} from LOCAL cache: {local_path} (quantize={quantize})")
    else:
        source = model_id
        from_local = False
        print(f"[{device.upper()}] Loading {model_id} from HuggingFace (quantize={quantize})")

    bnb_config = None
    if quantize == "4bit":
        bnb_config = BitsAndBytesConfig(
                                        load_in_4bit=True,
                                        bnb_4bit_quant_type="nf4",
                                        bnb_4bit_compute_dtype=torch.float16,
                                        bnb_4bit_use_double_quant=True,
                                        )
    elif quantize == "8bit":
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    try:
        tok = AutoTokenizer.from_pretrained(source, trust_remote_code=True)
        kwargs = dict(
                        device_map=device,
                        trust_remote_code=True,
                        low_cpu_mem_usage=True,
                    )
        if bnb_config is not None:
            kwargs["quantization_config"] = bnb_config
        else:
            kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32

        try:
            model = AutoModelForCausalLM.from_pretrained(source, **kwargs)
        except Exception as e:
            # Druhá poistka: ak samotné načítanie s kvantizáciou zlyhá, skús bez nej
            if bnb_config is not None:
                print(f"[QUANT] Quantized load failed: {e}")
                print("[QUANT] Retrying without quantization (fp16)...")
                kwargs.pop("quantization_config", None)
                kwargs["dtype"] = torch.float16 if device.startswith("cuda") else torch.float32
                model = AutoModelForCausalLM.from_pretrained(source, **kwargs)
                bnb_config = None
            else:
                raise

        if not from_local and bnb_config is None:
            try:
                os.makedirs(local_path, exist_ok=True)
                tok.save_pretrained(local_path)
                model.save_pretrained(local_path)
                print(f"[SAVE] Model cached to {local_path}")
            except Exception as e:
                print(f"[WARN] Could not save model locally: {e}")
        elif not from_local and bnb_config is not None:
            try:
                os.makedirs(local_path, exist_ok=True)
                tok.save_pretrained(local_path)
                print(f"[SAVE] Tokenizer cached to {local_path} (model skipped – quantized)")
            except Exception as e:
                print(f"[WARN] Could not save tokenizer locally: {e}")

        return pipeline("text-generation", model=model, tokenizer=tok)
    except Exception as e:
        print(f"[ERROR] Load failed ({device}, {quantize}): {e}")
        return None

# 3. RAG MANAGER
class RAGManager:
    def __init__(self):
        print("[EMB] Loading embeddings on CPU...")
        self.embeddings = HuggingFaceEmbeddings(model_name=CONFIG["emb_model_id"], model_kwargs={"device": "cpu"})
        self.vs = Chroma(collection_name="sk_legal_docs",persist_directory=CONFIG["rag_dir"],embedding_function=self.embeddings,)
        self.splitter = RecursiveCharacterTextSplitter(chunk_size=CONFIG["chunk_size"], chunk_overlap=CONFIG["chunk_overlap"])

    def add_text(self, text: str, metadata: dict):
        docs = self.splitter.create_documents([text], metadatas=[metadata])
        if docs:
            self.vs.add_documents(docs)

    def search(self, query: str) -> str:
        docs = self.vs.similarity_search(query, k=CONFIG["top_k"])
        return "\n\n".join(d.page_content for d in docs)

    def ingest_web(self, urls: List[str]):
        valid = [u for u in urls if any(d in u for d in CONFIG["allowed_domains"])]
        if not valid:
            return
        try:
            docs = self.splitter.split_documents(WebBaseLoader(valid).load())
            self.vs.add_documents(docs)
            print(f"[WEB] Ingested {len(docs)} chunks.")
        except Exception as e:
            print(f"[WARN] Web ingest failed: {e}")

# 4. PIPELINE

# a. Load and process text files in the directory
LOADERS = {".docx": Docx2txtLoader, ".rtf": load_rtf, ".txt": lambda p: TextLoader(p, encoding="utf-8")}

def read_file(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    loader = LOADERS.get(ext)
    if loader is None:
        return ""
    try:
        docs = loader(path).load() if ext != ".rtf" else loader(path)
        return "\n".join(d.page_content for d in docs)
    except Exception as e:
        print(f"[ERR] File load failed: {e}")
        return ""

def process_file(filepath: str, rag: RAGManager, summarizer, generator, terms: List[str]):
    filename = os.path.basename(filepath)
    print(f"Processing: {filename}")

    text = read_file(filepath)
    if not text.strip():
        return None

    rag.add_text(text, {"file": filename})

    # b. Summarization model on (CPU)
    summ_prompt = (
        f"Zhrň nasledovný text do stručného odstavca (max 5 viet). Píš po slovensky.\n\n"
        f"TEXT:\n{text[:12000]}\n\nZHRNUTIE:"
    )
    summary = run_llm(summarizer, summ_prompt, "ZHRNUTIE:") or text[:500]

    # c. RAG + generate output based on prompt
    matched = match_terms(text, terms)
    ctx = rag.search(f"{summary} {matched}")

    mode = CONFIG["mode"]
    main_prompt = "Navrhni JEDEN presný právny nadpis (Title)" if mode == "title" else "Vyber JEDEN kľúčový pojem (Keyword)"
    suffix = "Nadpis:" if mode == "title" else "Kľúčový pojem:"

    gen_prompt = (
        f"Kontext:\n{ctx[:2000]}\n\n"
        f"Zadanie: Na základe súhrnu a pojmov ({matched}) {main_prompt}.\n\n"
        f"Súhrn:\n{summary}\n\n{suffix}"
    )
    raw = run_llm(generator, gen_prompt, suffix)
    output = raw.split("\n")[0].strip('" ') if raw else "ERROR"

    return {
            "file": filename,
            "mode": mode,
            "summary": summary,
            "priority_terms": matched,
            "generated_output": output,
            }

# 5. MAIN FUNCTION
def main():
    if USE_CUDA:
        torch.cuda.empty_cache()
        
    print(f"STARTING PIPELINE (Mode: {CONFIG['mode']})")
    print("Hardware: RTX 4070 Super (Optimizing for 12GB VRAM)")

    stopwords = load_stopwords(CONFIG["stopwords_path"])
    terms = load_thesaurus(CONFIG["thesaurus_path"], CONFIG["thesaurus_col"], stopwords)

    summarizer = load_model(CONFIG["summ_model_id"], "cpu")
    generator = load_model(CONFIG["main_model_id"], "cuda:0", quantize="8bit")
    if not summarizer or not generator:
        print("[FATAL] Models failed to load.")
        return

    rag = RAGManager()
    rag.ingest_web(["https://www.slov-lex.sk/pravne-predpisy/SK/ZZ/2005/300/"])

    files = [f for f in os.listdir(CONFIG["input_dir"]) if f.lower().endswith((".docx", ".rtf", ".txt"))]
    results = [
                r for f in files
                if (r := process_file(os.path.join(CONFIG["input_dir"], f), rag, summarizer, generator, terms))
                ]

    if results:
        df = pd.DataFrame(results)
        out_name = f"output_{CONFIG['mode']}_{datetime.now():%Y-%m-%d}.csv"
        df.to_csv(os.path.join(CONFIG["output_dir"], out_name), index=False, encoding="utf-8-sig")
        print(f"\n[DONE] Saved to {out_name}")
        print(df[["file", "generated_output"]].head())
    
    if USE_CUDA:
        torch.cuda.empty_cache() 

if __name__ == "__main__":
    main()

IndentationError: expected an indented block after 'if' statement on line 248 (2331047365.py, line 249)